In [2]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch.nn as nn
%matplotlib inline

In [3]:
%%capture
! [ -f names.txt ] || wget https://raw.githubusercontent.com/karpathy/makemore/refs/heads/master/names.txt

In [4]:
df = pd.read_csv("names.txt", header=None)
names = df[0].str.lower()

In [5]:
context_size = 3
X, Y = [], []

for name in names:
	context = [0] * context_size
	for c in name:
		idx = ord(c) - ord("a")
		X.append(context)
		Y.append(idx)
		context = context[1:] + [idx]
	X.append(context)
	Y.append(26)

X = torch.tensor(X)
Y = torch.tensor(Y)

N = len(names)
Ntrain = N * 8 // 10

Xtrain, Ytrain = X[:Ntrain], Y[:Ntrain]
Xval, Yval = X[Ntrain:], Y[Ntrain:]

starting here, as expanded / ugly as possible

In [6]:
embedding_size = 3
nhidden = 200
nin = embedding_size*context_size

EMB = torch.randn(27, embedding_size)

W1 = torch.randn((nin,nhidden)) * (5/3) / nin**0.5
b1 = torch.randn(nhidden) * 0.1

W2 = torch.randn((300,27)) * 0.1
b2 = torch.randn(27) * 0.1

bngain = torch.randn((1, nhidden))*0.1 + 1.0
bnbias = torch.randn((1, nhidden))*0.1

params = [EMB, W1, b1, W2, b2, bngain, bnbias]

for p in params:
	p.requires_grad = True

forward pass

In [ ]:
n = 32
ixs = torch.randint(0, Xtrain.shape[0], (n,))

ctx, target = Xtrain[ixs], Ytrain[ixs]
emb = EMB[ctx]
embcat.view(-1, nin)

# Linear 1
h_pre_bn = embcat @ W1 + b1

# BatchNorm 1
bn_mean_i = h_pre_bn.sum(0, keepdim=True) / n
bn_diff = h_pre_bn - bn_mean_i
bn_diff2 = bn_diff**2
bn_var = bn_diff2.sum(0, keepdim=True) / (n-1)
bn_var_inv = (bn_var + 1e-5)**0.5
bn_raw = bn_diff * bn_var_inv
h_pre_act = bngain * bn_raw + bnbias

# Tanh
h = torch.tanh(h_pre_act)

# Linear 2
logits = h @ W2 + b2

# Cross Entropy
logits_max = logits.max(1, keepdim=True).values
norm_logits = logits - logits_max
counts = norm_logits.exp()
counts_sum = counts.sum(1, keepdim=True)
counts_sum_inv = counts_sum**-1
probs = counts * counts_sum_inv
logprobs = probs.log()
loss = -logprobs[range(n), target].mean()

backwards pass, line by line

In [ ]:
# Cross Entropy
dloss = torch.ones_like(loss)
dlogprobs = torch.zeros_like(logprobs)
dlogprobs[range(n), target] = - 1/n
dprobs = dlogprobs * (1 / probs)
dcounts_sum_inv = dprobs * counts
dcounts_sum = dcounts_sum_inv * (-counts_sum**-2)
dcounts = dcounts_sum * 1
dcounts += dprobs * counts_sum_inv
dnorm_logits = dcounts * counts
dlogits_max = dnorm_logits * -1

# Linear 2
dlogits = dnorm_logits * 1
dlogits += dlogits_max * F.one_hot(logits.max(1).indices, num_classes=logits.shape[1])

# Tanh
dh = dlogits @ W2.T
dW2 = dlogits * h
db2 = dlogits * 1

# BatchNorm
dh_pre_act = dh * (1 - h**2)
dbngain = dh_pre_act * bn_raw
dbnbias = dh_pre_act * 1
dbn_raw = dh_pre_act * bngain

dbn_var_inv = dbn_raw * bn_diff
dbn_var = dbn_var_inv * 0.5 * (bn_var + 1e-5)**-0.5
dbn_diff2 = torch.ones_like(bn_diff2) / (n-1)
dbn_diff = dbn_diff2 * 2 * bn_diff
dbn_diff += dbn_raw * bn_var_inv
dbn_mean_i = dbn_diff * -1

# Linear 
dh_pre_bn = torch.ones_like(h_pre_bn) / n
dh_pre_bn += bn_diff * 1

dembcat = dh_pre_bn @ W1.T
dW1 = dh_pre_bn @ embcat
db1 = dh_pre_bn * 1

demb = dembcat * 1

pytorch backwards pass

In [8]:
# PyTorch backward pass
for p in mymodel.parameters():
  p.grad = None
for t in [logprobs, probs, counts, counts_sum, counts_sum_inv,
          norm_logits, logit_max, logits, h, h_pre_act, bn_raw,
         bn_var_inv, bn_var, bn_diff2, bn_diff, h_pre_bn, bn_mean_i,
         embcat, emb]:
  t.retain_grad()
loss.backward()
loss

NameError: name 'mymodel' is not defined

In [ ]:
cmp('logprobs', dlogprobs, logprobs)
cmp('probs', dprobs, probs)
cmp('counts_sum_inv', dcounts_sum_inv, counts_sum_inv)
cmp('counts_sum', dcounts_sum, counts_sum)
cmp('counts', dcounts, counts)
cmp('norm_logits', dnorm_logits, norm_logits)
cmp('logit_maxes', dlogit_maxes, logit_maxes)
cmp('logits', dlogits, logits)
cmp('h', dh, h)
cmp('W2', dW2, W2)
cmp('b2', db2, b2)
cmp('hpreact', dhpreact, hpreact)
cmp('bngain', dbngain, bngain)
cmp('bnbias', dbnbias, bnbias)
cmp('bnraw', dbnraw, bnraw)
cmp('bnvar_inv', dbnvar_inv, bnvar_inv)
cmp('bnvar', dbnvar, bnvar)
cmp('bndiff2', dbndiff2, bndiff2)
cmp('bndiff', dbndiff, bndiff)
cmp('bnmeani', dbnmeani, bnmeani)
cmp('hprebn', dhprebn, hprebn)
cmp('embcat', dembcat, embcat)
cmp('W1', dW1, W1)
cmp('b1', db1, b1)
cmp('emb', demb, emb)
cmp('C', dC, C)

condensing CrossEntropyLoss

In [ ]:
# before:
# logit_maxes = logits.max(1, keepdim=True).values
# norm_logits = logits - logit_maxes # subtract max for numerical stability
# counts = norm_logits.exp()
# counts_sum = counts.sum(1, keepdims=True)
# counts_sum_inv = counts_sum**-1 # if I use (1.0 / counts_sum) instead then I can't get backprop to be bit exact...
# probs = counts * counts_sum_inv
# logprobs = probs.log()
# loss = -logprobs[range(n), Yb].mean()

dlogits = F.softmax(logits, 1)
dlogits[range(n), Yb] -= 1
dlogits /= n

cmp('logits', dlogits, logits)

In [ ]:
plt.figure(figsize=(4, 4))
plt.imshow(dlogits.detach(), cmap='gray')

batchnorm condensed

In [ ]:
# before:
# bnmeani = 1/n*hprebn.sum(0, keepdim=True)
# bndiff = hprebn - bnmeani
# bndiff2 = bndiff**2
# bnvar = 1/(n-1)*(bndiff2).sum(0, keepdim=True) # note: Bessel's correction (dividing by n-1, not n)
# bnvar_inv = (bnvar + 1e-5)**-0.5
# bnraw = bndiff * bnvar_inv
# hpreact = bngain * bnraw + bnbias

hpreact_fast = bngain * (hprebn - hprebn.mean(0, keepdim=True)) / torch.sqrt(hprebn.var(0, keepdim=True, unbiased=True) + 1e-5) + bnbias
print('max diff:', (hpreact_fast - hpreact).abs().max())

In [ ]:
# before we had:
# dbnraw = bngain * dhpreact
# dbndiff = bnvar_inv * dbnraw
# dbnvar_inv = (bndiff * dbnraw).sum(0, keepdim=True)
# dbnvar = (-0.5*(bnvar + 1e-5)**-1.5) * dbnvar_inv
# dbndiff2 = (1.0/(n-1))*torch.ones_like(bndiff2) * dbnvar
# dbndiff += (2*bndiff) * dbndiff2
# dhprebn = dbndiff.clone()
# dbnmeani = (-dbndiff).sum(0)
# dhprebn += 1.0/n * (torch.ones_like(hprebn) * dbnmeani)

dhprebn = bngain*bnvar_inv/n * (n*dhpreact - dhpreact.sum(0) - n/(n-1)*bnraw*(dhpreact*bnraw).sum(0))

cmp('hprebn', dhprebn, hprebn)